# Subsample analysis — Alexi UC (`metab@` per-gene-pair permutations)

Reads a **subsampling run** (`spacetravlr_subsamples/run_{R}/`) produced by
`metab_processing/SpaceTravLR/run_subsamples.py` and, for a chosen target gene and cell
type (start with **`T`**), computes:

1. **average β** per gene pair (`metab@{Metabolite}-{g1}_{g2}`) within the cell type,
2. the **distribution** of β for each gene pair, and
3. the **R²** between the cell type's raw expression of the target gene and **β·x**.

Each surviving gene pair is its own `metab@…` column (both orientations summed), so here a
"gene pair" *is* one metabolite column. Companion to `alexi_uc_NDRG1.ipynb` (the full-panel
run); the difference is that a subsample run has one mini-fit per draw, and no pre-saved
`spacetravlr_adata.h5ad`, so we attach β / x / β·x on the fly.

> **Why `subsample_beta_means.csv` came out with only a header (`0 rows`).** The fit *did*
> train metabolites (the training log shows 1-4 per subsample), so the betadata carries
> `beta_metab@...` columns. The empty CSV comes from Object B, `beta_analysis.tier_means(...,
> group="metab")`: it groups each trained cell by `obs[cell_type_col]`, and `groupby` **drops
> NaN groups** -- so if the trained cells carry **no label** under that annotation column,
> every group is dropped and it writes the header-only fallback -> `0 rows`. (Object A,
> `subsample_betas.h5ad`, still reports "186 obsm matrices" because it reindexes each parquet
> onto all cells without grouping.) This is **not** empty metabolites and **not** a barcode
> mismatch: a barcode mismatch would raise `KeyError`, and all-NaN betas would still yield
> rows -- only *unlabeled trained cells* empty it silently. The **Diagnostic** below checks the
> annotation coverage of the trained cells directly (part c is the decisive one); the pipeline
> now logs a WARNING naming how many trained cells carry a label.

In [ ]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine (mirrors alexi_uc_NDRG1.ipynb).
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import json
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import linregress

from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.SpaceTravLR import beta_analysis
from metab_processing.SpaceTravLR.dataset_configs import dataset_paths, get_config
from metab_processing.SpaceTravLR.metab_loader import load_metabolite_selection
from metab_processing.SpaceTravLR.run_subsamples import pairs_to_metabolites, latest_run

In [ ]:
# ============================ SETTINGS ============================
PROJECT   = 'Alexi_UC_Spliced'
DATASET   = '13473_HS4_UC-Slice_1'   # which slice's subsample run to read
RUN       = -1                       # subsampling run number; -1 = latest
TIER      = '25_06_11_ICI_5K_Coarse_annotations'   # cell-type annotation column
CELL_TYPE = 'T'                      # start here (Foster's ask)
GENE      = 'NDRG1'                  # primary target gene for the single-gene cells

DATA_DIR_PROJ = f'{DATA_DIR}/{PROJECT}'
paths = dataset_paths(DATASET, DATA_DIR_PROJ)
cfg   = get_config(DATASET)
FOCUS_GENES = cfg['focus_genes']

dataset_dir  = paths['dataset_dir']
harr_dir     = paths['selection_yaml'].parent                 # easy_download/harreman_outputs
subs_yaml_root = harr_dir / 'subsamples'                      # the sampled ymls live here
sub_root_all   = dataset_dir / 'spacetravlr_subsamples'       # the fitted betas live here

R = latest_run(sub_root_all) if RUN == -1 else RUN
sub_root = sub_root_all / f'run_{R}'
yaml_dir = subs_yaml_root / f'run_{R}'
print('dataset :', DATASET)
print('run     :', R, '->', sub_root)
assert sub_root.is_dir(), f'no fitted run at {sub_root} (available: {sorted(p.name for p in sub_root_all.glob("run_*"))})'

## Diagnostic — did the fit actually train any `metab@` columns?

Run this first. It answers the "0 rows" question on *your* data by (a) checking the base
panel's transporter symbols against the trained panel's `var_names`, and (b) counting the
`beta_metab@…` columns in a real betadata parquet from this run.

In [ ]:
import pyarrow.parquet as pq

# (a) base sample_metabolites.yml gene symbols vs the trained panel's var_names
base_yml = harr_dir / 'sample_metabolites.yml'
processed_adata = sub_root / '_setup' / 'spacetravlr_output' / 'input_data' / '_adata.h5ad'
print('base panel yml :', base_yml, '(exists:', base_yml.is_file(), ')')
print('processed adata:', processed_adata, '(exists:', processed_adata.is_file(), ')')

var_names = None
if processed_adata.is_file():
    _b = sc.read_h5ad(processed_adata, backed='r')
    var_names = set(map(str, _b.var_names))
    if _b.isbacked:
        _b.file.close()

if base_yml.is_file() and var_names is not None:
    base_sel = load_metabolite_selection(base_yml)
    base_genes = {g for pairs in base_sel.values() for pair in pairs for g in pair}
    missing = sorted(base_genes - var_names)
    present = sorted(base_genes & var_names)
    print(f'\nbase transporter genes: {len(base_genes)} | in panel: {len(present)} | MISSING: {len(missing)}')
    if missing:
        print('  MISSING from var_names (these get var-filtered out -> no metab@ columns):')
        print('   ', missing)
    else:
        print('  all base transporter genes are in the panel ✓')

# (b) count metab@ columns in one real betadata parquet from this run
_pq = sorted(sub_root.glob('subsample_*/spacetravlr_output/betadata/*_betadata.parquet'))
print(f'\nbetadata parquets found: {len(_pq)}')
if _pq:
    names = [c[len('beta_'):] for c in pq.read_schema(_pq[0]).names if c.startswith('beta_')]
    metab = [n for n in names if beta_analysis._group(n) == 'metab']
    print(f'example parquet: {_pq[0].relative_to(sub_root)}')
    print(f'  total beta_ columns: {len(names)} | metab@ columns: {len(metab)}')
    print('  example metab@ cols:', metab[:6] if metab else '(NONE -> this is why the CSV is header-only)')

# (c) THE DECISIVE CHECK: do the trained cells carry a label under cell_type_col? tier_means
# groups by this and drops NaN groups, so all-unlabeled trained cells -> header-only CSV.
if _pq:
    _cell_type_col = cfg['cell_type_src']            # what build_run_analysis grouped by
    _bidx = beta_analysis._read_betas(_pq[0], group='metab').index
    _obs = sc.read_h5ad(paths['adata'], backed='r').obs
    _in = _bidx.intersection(_obs.index)
    print(f'\ntrained cells in this parquet: {len(_bidx)} | also in display adata.obs: {len(_in)}')
    if len(_in):
        _lab = _obs.loc[_in, _cell_type_col]
        print(f'grouped by cell_type_col={_cell_type_col!r}: labeled {int(_lab.notna().sum())}/{len(_in)}')
        print('  label counts (dropna=False):')
        print(_lab.value_counts(dropna=False).head(12).to_string())
        if _lab.notna().sum() == 0:
            print('  -> ALL trained cells are unlabeled here: this is why the CSV is empty. '
                  'Use an annotation column that covers these cells.')

    # Reproduce build_run_analysis's Object B exactly (same obs source, same column, same call)
    # so this notebook's verdict matches the SLURM run's CSV row count.
    _bd = _pq[0].parent
    _rows = beta_analysis.tier_means(_bd, _obs,
                                     tier=_cell_type_col, group='metab')
    print(f'\ntier_means rows for {_bd.parent.parent.name} (what the CSV gets): {len(_rows)}')
    if len(_rows) == 0:
        print('  -> CONFIRMED: this reproduces the empty subsample_beta_means.csv.')

In [ ]:
# The shared metabolite communication score `x` is target-gene-independent and identical for a
# given gene pair across subsamples, so we compute it ONCE for the UNION of every pair that
# appears in the run (one diffusion pass), then slice per subsample. This mirrors the full
# run's artifacts stage (run_spacetravlr.py) but avoids re-diffusing per draw.

def _subsample_indices(sub_root):
    idx = []
    for d in sub_root.glob('subsample_*'):
        try:
            idx.append(int(d.name.rsplit('_', 1)[1]))
        except ValueError:
            pass
    return sorted(idx)

def _selection_j(j):
    return load_metabolite_selection(yaml_dir / f'sampled_metabolites_{j}.yml')

def _run_params():
    # radius / contact_distance / scale_factor / layer are shared across subsamples (one setup,
    # one fit_kwargs). Read the first available run_params.json.
    for p in sorted(sub_root.glob('subsample_*/spacetravlr_output/betadata/run_params.json')):
        rp = json.loads(p.read_text())
        return dict(radius=rp['radius'],
                    contact_distance=rp.get('contact_distance', 50),
                    scale_factor=rp.get('scale_factor', 100),
                    layer=rp.get('layer', 'imputed_count'))
    raise FileNotFoundError('no run_params.json under any subsample betadata/')

SAMPLES = _subsample_indices(sub_root)
print('subsamples:', SAMPLES)

In [ ]:
# Load once: the display adata (cell types + RAW counts in .X) and the processed adata that the
# diffusion for x must run over (has the training `layer` + obsm['spatial']).
adata = sc.read_h5ad(paths['adata'])
adata.uns['sample'] = DATASET
assert TIER in adata.obs, f'{TIER!r} not in adata.obs; columns e.g. {list(adata.obs.columns)[:10]}'
print('display adata:', adata.shape, '| cell types:', sorted(map(str, adata.obs[TIER].unique())))

x_adata = sc.read_h5ad(processed_adata)
X_PARAMS = _run_params()
print('x params:', X_PARAMS)

# Union of all gene-pair "metabolite" columns across the run (var-filtered like the fit did).
union_meta = {}
for j in SAMPLES:
    union_meta.update(pairs_to_metabolites(_selection_j(j), var_names=list(x_adata.var_names)))
print('union gene-pair columns across the run:', len(union_meta))

# ONE diffusion pass -> cells x union-pair-columns (metab@... names), aligned to display cells.
if union_meta:
    X = beta_analysis.compute_metab_x(x_adata, union_meta, **X_PARAMS).reindex(adata.obs_names)
    X.columns = [f'metab@{c}' if not c.startswith('metab@') else c for c in X.columns]
else:
    X = pd.DataFrame(index=adata.obs_names)
print('x_metab table:', X.shape)

In [ ]:
# Per subsample, per focus gene: the metab betas (cells x pair-columns), aligned to display
# cells. BETAS[j][gene] is a DataFrame indexed by obs_names, columns = 'metab@...'.
BETAS = {}
for j in SAMPLES:
    bdir = sub_root / f'subsample_{j}' / 'spacetravlr_output' / 'betadata'
    per_gene = {}
    for gene in FOCUS_GENES:
        pqf = bdir / f'{gene}_betadata.parquet'
        if not pqf.is_file():
            continue
        b = beta_analysis._read_betas(pqf, group='metab').reindex(adata.obs_names)
        if not b.columns.empty:
            per_gene[gene] = b
    if per_gene:
        BETAS[j] = per_gene

n_pairs_total = sum(b.shape[1] for g in BETAS.values() for b in g.values())
print(f'loaded betas for {len(BETAS)} subsamples; total (gene x pair) metab columns: {n_pairs_total}')
if n_pairs_total == 0:
    print('\n*** No metab@ betas in this run — see the Diagnostic above. The analyses below '
          'will be empty until sample_metabolites.yml gene symbols match the panel and the run '
          'is re-fit. ***')

## 1. Average β per gene pair within a cell type

One row per `(subsample, gene, gene_pair)` with the mean/std β over the chosen cell type's
cells, plus a cross-subsample summary (a gene pair recurs across draws whenever the Bernoulli
sample kept it). This is exactly what a *populated* `subsample_beta_means.csv` holds; we
compute it straight from the betadata so it is correct regardless of that file.

In [ ]:
def _ct_mask(cell_type):
    return (adata.obs[TIER].astype(str) == cell_type).to_numpy()

def avg_beta_by_pair(gene, cell_type=CELL_TYPE):
    """Mean/std β for `gene` per gene pair within `cell_type`, one row per (subsample, pair)."""
    mask = _ct_mask(cell_type)
    rows = []
    for j in SAMPLES:
        b = BETAS.get(j, {}).get(gene)
        if b is None:
            continue
        sub = b.loc[mask]
        for pair in sub.columns:
            v = sub[pair].to_numpy()
            v = v[~np.isnan(v)]
            if v.size:
                rows.append({'sample': j, 'gene': gene, 'gene_pair': pair,
                             'mean_beta': float(v.mean()), 'std_beta': float(v.std()),
                             'n_cells': int(v.size)})
    return pd.DataFrame(rows)

def avg_beta_summary(gene, cell_type=CELL_TYPE):
    """Collapse the per-subsample means to one row per gene pair (mean of subsample means)."""
    df = avg_beta_by_pair(gene, cell_type)
    if df.empty:
        return df
    g = df.groupby('gene_pair')
    out = pd.DataFrame({
        'n_subsamples': g.size(),
        'mean_beta': g['mean_beta'].mean(),          # avg over draws of the per-draw cell mean
        'std_over_subsamples': g['mean_beta'].std(),  # stability across draws
        'mean_abs_beta': g['mean_beta'].apply(lambda s: np.mean(np.abs(s))),
    }).reset_index().sort_values('mean_abs_beta', ascending=False, ignore_index=True)
    return out

print(f'per-(subsample, pair) means for {GENE} in {CELL_TYPE!r}:')
display(avg_beta_by_pair(GENE, CELL_TYPE).head(20))
print(f'\ncross-subsample summary for {GENE} in {CELL_TYPE!r}:')
display(avg_beta_summary(GENE, CELL_TYPE).head(20))

In [ ]:
# The pipeline ALSO writes this as `subsample_beta_means.csv` (Object B of build_run_analysis)
# -- `tier_means` per subsample. It is the "already calculated" version of the table above.
# It is header-only whenever the run trained no metab@ columns (see the Diagnostic); once the
# gene symbols in sample_metabolites.yml match the panel and the run is re-fit, it matches
# `avg_beta_by_pair` for group='metab'.
_csv = sub_root / 'subsample_beta_means.csv'
if _csv.is_file():
    _means = pd.read_csv(_csv)
    print(f'{_csv.name}: {len(_means)} rows')
    _f = _means[(_means.get('gene') == GENE) & (_means.get('cell_type').astype(str) == CELL_TYPE)]
    display(_f.head(20) if len(_means) else _means.head())
else:
    print('no subsample_beta_means.csv yet at', _csv)

## 2. Distribution of β for each gene pair

For the chosen target gene and cell type, the per-cell β distribution of each gene pair,
pooled over the cell type's cells and over every subsample that kept the pair. Overlaid
histograms (one per pair); pass `top=` to limit to the strongest pairs by mean |β|.

In [ ]:
def beta_values_by_pair(gene, cell_type=CELL_TYPE):
    """{gene_pair: pooled per-cell β array over cell_type cells across all subsamples}."""
    mask = _ct_mask(cell_type)
    pooled = {}
    for j in SAMPLES:
        b = BETAS.get(j, {}).get(gene)
        if b is None:
            continue
        sub = b.loc[mask]
        for pair in sub.columns:
            v = sub[pair].to_numpy()
            v = v[~np.isnan(v)]
            pooled.setdefault(pair, []).append(v)
    return {k: np.concatenate(v) for k, v in pooled.items() if len(v)}

def plot_beta_distributions(gene, cell_type=CELL_TYPE, top=None, bins=60):
    pooled = beta_values_by_pair(gene, cell_type)
    if not pooled:
        print(f'no gene-pair betas for {gene} in {cell_type!r}'); return
    order = sorted(pooled, key=lambda k: -np.mean(np.abs(pooled[k])))
    if top:
        order = order[:top]
    plt.figure(figsize=(9, 5))
    for pair in order:
        plt.hist(pooled[pair], bins=bins, alpha=0.45,
                 label=pair.replace('metab@', ''))
    plt.axvline(0, color='k', lw=0.6)
    plt.xlabel(f'{gene} β'); plt.ylabel('cells (pooled over subsamples)')
    plt.title(f'{DATASET} | {gene} β distribution per gene pair | {cell_type}')
    plt.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

plot_beta_distributions(GENE, CELL_TYPE, top=12)

## 3. R² — cell-type raw expression vs β·x

For each gene pair, β·x is that pair's contribution to the model for the target gene
(`β_gene[pair] · x[pair]`). We regress the cell type's **raw counts** of the target gene on
β·x and report R² — computed per subsample, then averaged across subsamples per gene pair.
Mirrors `betax_vs_gene_r2` in `alexi_uc_NDRG1.ipynb`, with the y-target fixed to raw counts.

In [ ]:
def _raw_counts(gene, mask):
    """Raw counts of `gene` for the masked cells (display adata .X is raw counts)."""
    xg = adata[:, gene].X
    xg = xg.toarray().ravel() if hasattr(xg, 'toarray') else np.asarray(xg).ravel()
    return xg[mask]

def _r2(a, b):
    m = ~np.isnan(a) & ~np.isnan(b)
    if m.sum() < 3 or np.ptp(a[m]) == 0 or np.ptp(b[m]) == 0:
        return np.nan, int(m.sum())
    return float(linregress(a[m], b[m]).rvalue ** 2), int(m.sum())

def betax_r2_by_pair(gene, cell_type=CELL_TYPE):
    """One row per (subsample, gene_pair): R² of raw counts vs β·x for `gene` in `cell_type`."""
    if gene not in adata.var_names:
        print(f'{gene} not in adata.var_names'); return pd.DataFrame()
    mask = _ct_mask(cell_type)
    raw = _raw_counts(gene, mask)
    rows = []
    for j in SAMPLES:
        b = BETAS.get(j, {}).get(gene)
        if b is None:
            continue
        bsub = b.loc[mask]
        for pair in bsub.columns:
            if pair not in X.columns:
                continue
            betax = bsub[pair].to_numpy() * X.loc[mask, pair].to_numpy()
            r2, n = _r2(betax, raw)
            rows.append({'sample': j, 'gene': gene, 'gene_pair': pair, 'r2': r2,
                         'avg_betax': float(np.nanmean(betax)),
                         'avg_abs_beta': float(np.nanmean(np.abs(bsub[pair].to_numpy()))),
                         'avg_x': float(np.nanmean(X.loc[mask, pair].to_numpy())),
                         'avg_raw_expr': float(np.nanmean(raw)), 'n_cells': n})
    return pd.DataFrame(rows)

def betax_r2_summary(gene, cell_type=CELL_TYPE):
    df = betax_r2_by_pair(gene, cell_type)
    if df.empty:
        return df
    g = df.groupby('gene_pair')
    return (pd.DataFrame({'n_subsamples': g.size(),
                          'mean_r2': g['r2'].mean(),
                          'std_r2': g['r2'].std(),
                          'mean_avg_betax': g['avg_betax'].mean()})
            .reset_index().sort_values('mean_r2', ascending=False, ignore_index=True))

print(f'per-(subsample, pair) R² for {GENE} raw counts vs β·x in {CELL_TYPE!r}:')
display(betax_r2_by_pair(GENE, CELL_TYPE).sort_values('r2', ascending=False, ignore_index=True).head(20))
print(f'\ncross-subsample R² summary for {GENE} in {CELL_TYPE!r}:')
display(betax_r2_summary(GENE, CELL_TYPE).head(20))

In [ ]:
# Scatter for one (gene, gene_pair, subsample): raw counts vs β·x, with the R² fit line.
def plot_betax_vs_expr(gene, gene_pair, sample=None, cell_type=CELL_TYPE):
    if not gene_pair.startswith('metab@'):
        gene_pair = 'metab@' + gene_pair
    js = [sample] if sample is not None else [j for j in SAMPLES
                                              if gene_pair in BETAS.get(j, {}).get(gene, pd.DataFrame()).columns]
    if not js:
        print(f'{gene_pair} not fit for {gene} in any listed subsample'); return
    mask = _ct_mask(cell_type)
    raw = _raw_counts(gene, mask)
    n = len(js); ncol = min(3, n); nrow = -(-n // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4 * nrow), squeeze=False)
    for ax, j in zip(axes.ravel(), js):
        b = BETAS[j][gene].loc[mask, gene_pair].to_numpy()
        betax = b * X.loc[mask, gene_pair].to_numpy()
        r2, ncell = _r2(betax, raw)
        ax.scatter(betax, raw, s=10, alpha=0.5)
        m = ~np.isnan(betax) & ~np.isnan(raw)
        if m.sum() >= 3 and np.ptp(betax[m]) > 0:
            s, i, *_ = linregress(betax[m], raw[m])
            xs = np.linspace(betax[m].min(), betax[m].max(), 50)
            ax.plot(xs, s * xs + i, 'r--', label=f'R²={r2:.3f} (n={ncell})')
            ax.legend(fontsize=8)
        ax.set_xlabel(f'{gene} β·x'); ax.set_ylabel(f'{gene} raw counts')
        ax.set_title(f'subsample {j}')
    for ax in axes.ravel()[n:]:
        ax.axis('off')
    fig.suptitle(f'{DATASET} | {gene} ~ {gene_pair.replace("metab@","")} | {cell_type}')
    fig.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

# Example: pick the strongest-mean-R² pair for GENE and plot it across subsamples.
_summ = betax_r2_summary(GENE, CELL_TYPE)
if not _summ.empty:
    plot_betax_vs_expr(GENE, _summ.iloc[0]['gene_pair'], cell_type=CELL_TYPE)